In [3]:
import pandas as pd

In [5]:
homework = pd.read_clipboard()

In [ ]:
homework_regrade = pd.read_clipboard()

In [ ]:

def merge_hw_with_regrade(hw_df, regrade_df,
                          file_col='file',
                          ppq_label='points-per-question',
                          key_len=7,
                          fillna_value=0,
                          export_path=None):
    """
    Merge two hw tables (hw_df first, regrade second) and:
      - Match students by first `key_len` chars of the filename prefix
      - Keep the hw_df row when duplicates exist
      - Preserve the 'points-per-question' row
      - Add `total` (sum of question columns) as second-to-last column
      - Add `missing_items` (list or comma-string of questions where score < ppq) as last column
      - Optionally export the result to Excel (export_path)
    Returns: final DataFrame
    """
    # work on copies
    a = hw_df.copy()
    b = regrade_df.copy()

    # --- 0. basic checks
    if file_col not in a.columns or file_col not in b.columns:
        raise ValueError(f"Both tables must have a '{file_col}' column")

    # --- 1. extract & remove points-per-question row(s) from both, but prefer hw_df's one
    ppq_a = a[a[file_col] == ppq_label]
    ppq_b = b[b[file_col] == ppq_label]

    # prefer hw_df's ppq if it exists, else take from regrade if present
    if not ppq_a.empty:
        ppq_row = ppq_a.iloc[[0]]  # keep as DataFrame
    elif not ppq_b.empty:
        ppq_row = ppq_b.iloc[[0]]
    else:
        raise ValueError(f"No '{ppq_label}' row found in either table")

    # remove any ppq rows from source tables to avoid duplication during dedupe
    a = a[a[file_col] != ppq_label].reset_index(drop=True)
    b = b[b[file_col] != ppq_label].reset_index(drop=True)

    # --- 2. create student_clean and key for matching
    def make_student_clean(s):
        # if filename doesn't have underscore, fallback to full filename
        if pd.isna(s):
            return ""
        return str(s).split('_')[0]

    a['student_clean'] = a[file_col].apply(make_student_clean)
    b['student_clean'] = b[file_col].apply(make_student_clean)

    a['key'] = a['student_clean'].str[:key_len].str.lower()
    b['key'] = b['student_clean'].str[:key_len].str.lower()

    # --- 3. unify columns (so both frames have same set of columns)
    all_cols = a.columns.union(b.columns)
    # keep original ordering as much as possible but include union
    a = a.reindex(columns=all_cols, fill_value=pd.NA)
    b = b.reindex(columns=all_cols, fill_value=pd.NA)

    # --- 4. concat with hw_df first so its rows are preferred when deduping
    combined = pd.concat([a, b], ignore_index=True)

    # --- 5. deduplicate by 'key', keeping the first occurrence (hw_df rows were first)
    deduped = combined.drop_duplicates(subset='key', keep='first').reset_index(drop=True)

    # --- 6. put the points-per-question row back at top
    # ensure ppq_row has same columns as deduped
    ppq_row = ppq_row.reindex(columns=deduped.columns, fill_value=pd.NA)
    result = pd.concat([ppq_row, deduped], ignore_index=True)

    # --- 7. identify question columns (starts with 'q')
    question_cols = [c for c in result.columns if str(c).startswith('q')]

    # coerce question columns to numeric for computations (errors -> NaN)
    for c in question_cols:
        result[c] = pd.to_numeric(result[c], errors='coerce')

    # --- 8. prepare comparison baseline from points-per-question row
    ppq_series = result.loc[result[file_col] == ppq_label, question_cols].iloc[0]
    # if there are NaNs in ppq (unlikely), treat them as 0 for comparison
    ppq_series = ppq_series.fillna(fillna_value)

    # --- 9. compute total (sum across question columns) using fillna_value for missing
    # We will calculate totals after filling missing question cell values with fillna_value for numeric sum
    result_totals = result[question_cols].fillna(fillna_value).sum(axis=1)
    result['total'] = result_totals

    # --- 10. compute missing_items: list of question names where student_score < ppq
    def missing_list(row):
        # for the ppq row itself return empty list
        if row[file_col] == ppq_label:
            return []
        misses = []
        for q in question_cols:
            student_val = row[q]
            # treat NaN as fillna_value when deciding if they missed the full points
            if pd.isna(student_val):
                student_val = fillna_value
            # compare numerically
            try:
                if float(student_val) < float(ppq_series[q]):
                    misses.append(q)
            except Exception:
                # if something odd (non-numeric), consider it missing
                misses.append(q)
        return misses

    result['missing_items'] = result.apply(missing_list, axis=1)

    # --- 11. format missing_items to string for easier Excel viewing (optional)
    # Keep both: leave lists in the DataFrame but create a display string column
    result['missing_items_str'] = result['missing_items'].apply(lambda lst: ','.join(lst) if lst else '')

    # --- 12. reorder columns so that 'total' is second-to-last and 'missing_items' (or str) is last
    # We'll put the list column (missing_items_str) as the final column for export/readability
    cols = [c for c in result.columns if c not in ('total', 'missing_items', 'missing_items_str')]
    final_cols = cols + ['total', 'missing_items_str']
    final = result.reindex(columns=final_cols)

    # --- 13. optionally fill remaining non-question NaNs (you can change behavior if desired)
    # If you prefer to turn all NAs to 0 for question cols only, uncomment:
    # final[question_cols] = final[question_cols].fillna(fillna_value)

    # --- 14. export to excel if path provided
    if export_path:
        # write missing_items_str (better for Excel). Do not write the list column.
        final.to_excel(export_path, index=False)

    return final


In [ ]:
# basic usage, return DataFrame (no export)
final = merge_hw_with_regrade(homework, homework_regrade)

# save to Excel
final = merge_hw_with_regrade(homework, homework_regrade,
                              export_path='merged_hw12.xlsx')


In [7]:

def analyze_hw_table(df):
    """
    df: a dataframe where
         - row 0 is the points-per-question row
         - column 0 is the student identifier
         - remaining rows are student scores
    """

    # ---- 1. Extract points-per-question row ----
    ppq = df.iloc[0]                   # whole row
    question_cols = df.columns[1:]     # all question columns (skip column 0)

    # ---- 2. Make a copy so we don't modify original ----
    out = df.copy()

    # ---- 3. Compute total score per student ----
    # skip row 0 (points row)
    out.loc[1:, "total"] = out.loc[1:, question_cols].sum(axis=1)

    # ---- 4. Create missing_questions column ----
    missing_list = []
    for idx in out.index:
        if idx == 0:     # keep PPQ row intact
            missing_list.append([])
            continue

        row = out.loc[idx, question_cols]
        missed = [q for q in question_cols if row[q] != ppq[q]]
        missing_list.append(missed)

    out["missing_questions"] = missing_list

    return out


In [9]:
result = analyze_hw_table(homework)
#result

In [11]:
result.to_excel("Homework_1_analyzed.xlsx", index=False)
